# 20 - Results analysis (local)
Loads the downloaded `out_A` and `out_B` folders and builds the tables, plots and galleries. Set the two paths below. Each section starts with a plain-words note on what it shows and how to read it.

In [ ]:
OUT_A = r'D:/Kaggle/out_A'   # unzipped output of notebook 11
OUT_B = r'D:/Kaggle/out_B'   # unzipped output of notebook 12
NOW_DATA = r'Data'            # local NoW data (only needed for scan-error heatmaps)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from IPython.display import Markdown
from src import analysis as an
from src.reporting import table_markdown
show = lambda rows: Markdown(table_markdown(rows))
tagged = an.load_mesh_records(OUT_A)
rows = an.score_rows(tagged, OUT_A)
print(len(tagged), 'attempts;', sum(r['mean_mm'] is not None for r in rows), 'scored')

## B1. Clean mesh error against the scan
Each row is one model on the original photos of the 20 people. Error is the distance from each scan point to the fitted face surface after the local alignment, in millimetres; lower is better. `within 2.5 mm` is the share of scan points that are close. Statistics are taken over people (each person counts once).

In [ ]:
show(an.clean_mesh_table(rows))

## B2. Degraded images: against the scan and against the clean result
For every model and condition, the mean error against the scan, then the paired change from the same person's clean fit (positive means worse). The Wilcoxon p-value tests whether the per-person changes are centred on zero; with 20 people, small p-values are suggestive rather than proof.

In [ ]:
deg = an.degradation_table(rows)
show(deg)

In [ ]:
an.plot_error_vs_severity(deg)

The curve shows error as the face gets smaller in the image (right to left means more severe). The compound condition is labelled; it is a different mix (blur, noise, JPEG) at 48 px width.

In [ ]:
deltas = an.per_person_deltas(rows)
an.plot_paired_lines(deltas, 'vits')

Each line is one person: how their error changes from clean to each degraded condition. Lines that mostly go up mean the damage is consistent; a tangle around zero means the effect is small compared with person-to-person scatter.

In [ ]:
an.plot_ced_by_condition(rows, 'vits')

Cumulative error curves pool every scored scan point. A curve further right is worse; the gap between curves shows how much the whole error distribution shifts.

## Real-world photos
Expressions, occlusions and selfies (originals only, no degradation) for all five models. Compare categories to see which kinds of photo hurt which model.

In [ ]:
show(an.real_world_table(rows))

## Oracle: how much could perfect priors recover?
ViT-S was re-run on degraded photos with the clean photo's priors swapped in. `gap recovered` = (degraded - oracle) / (degraded - clean). Near 1 means the priors are the whole problem; near 0 means a prior correction cannot help for that condition. When the clean-versus-degraded gap is tiny the ratio is unstable - read the gap column first.

In [ ]:
show(an.oracle_table(rows))

## B3. Clean appearance against the photo
For each method, the re-rendered textured face is compared with the input photo inside the face mask. PSNR and SSIM: higher is better. LPIPS, CIEDE2000 colour difference and ITA skin-tone angle difference: lower is better. ArcFace cosine: closer to 1 means the identity is kept. `unavailable` entries mean that metric's package or model was missing.

In [ ]:
results, pairs = an.load_appearance(OUT_B)
show(an.appearance_clean_table(results))

## B4. Degraded appearance against the same method's clean result
This asks how much each method's output moves when the photo gets worse. Render columns compare the images; UV columns compare the texture atlases on the region visible in both.

In [ ]:
app = an.appearance_degraded_table(pairs)
show(app)

In [ ]:
an.plot_appearance_curves(app, 'render ciede2000')

The curve tracks colour shift (CIEDE2000, lower is better) as the photo degrades; a flat line means the method is stable.

## Cost
One row per model: first-call time on a GPU (cold), median later calls (warm), peak GPU memory, counted GFLOPs and parameters where profiling worked. Each ViT fit runs in a fresh process, so even 'warm' includes model loading.

In [ ]:
import json
cost_rows = json.load(open(os.path.join(OUT_A, 'cost_summary.json')))
show(cost_rows)

In [ ]:
an.plot_cost_bars(cost_rows)

## Image galleries
Input strip and clay renders for one person across conditions, with scan-error heatmaps. Brighter heatmap colour means farther from the scan (clipped at 5 mm).

In [ ]:
import numpy as np
from PIL import Image
subject = sorted({r['subject'] for r in rows if r['kind'] == 'clean'})[0]
sel = [r for r in rows if r['kind'] == 'clean' and r['subject'] == subject and r['model'] == 'vits' and r['fit'] == 'success']
imgs, titles = [], []
for r in sorted(sel, key=lambda r: an.CONDITIONS.index(r['condition'])):
    obj = os.path.join(r['folder'], 'fit', 'neutral.obj')
    if os.path.isfile(obj):
        imgs.append(an.clay_render(obj)); titles.append(r['condition'])
an.show_image_grid(imgs, titles)

In [ ]:
# Scan-error heatmaps (needs the local NoW data to re-read the scan).
from src.data_now import installed_cases
import matplotlib.pyplot as plt
cases = {c.image_key: c for c in installed_cases(NOW_DATA)}
sel = [r for r in rows if r['kind'] == 'clean' and r['subject'] == subject and r['model'] == 'vits' and r['score_status'] == 'success']
sel = sorted(sel, key=lambda r: an.CONDITIONS.index(r['condition']))
fig, axes = plt.subplots(1, max(len(sel), 1), figsize=(3 * max(len(sel), 1), 3.2), squeeze=False)
for ax, r in zip(axes[0], sel):
    case = cases[r['image']]
    d = np.load(os.path.join(r['folder'], 'distances.npz'))['distances_mm']
    an.scan_error_heatmap(case.scan_path, case.scan_landmark_path, d, ax)
    ax.set_title(r['condition'], fontsize=8)

In [ ]:
# UV textures and clean-vs-degraded renders for the appearance methods.
import glob
method = 'raw_projection'
folders = sorted(glob.glob(os.path.join(OUT_A, 'appearance', method, subject, '*')))
imgs = [np.asarray(Image.open(os.path.join(f, 'uv_texture.png'))) for f in folders if os.path.isfile(os.path.join(f, 'uv_texture.png'))]
titles = [os.path.basename(f) for f in folders if os.path.isfile(os.path.join(f, 'uv_texture.png'))]
an.show_image_grid(imgs, titles) if imgs else print('no UV textures found')

In [ ]:
imgs = [np.asarray(Image.open(os.path.join(f, 'render.png'))) for f in folders if os.path.isfile(os.path.join(f, 'render.png'))]
titles = [os.path.basename(f) for f in folders if os.path.isfile(os.path.join(f, 'render.png'))]
an.show_image_grid(imgs, titles) if imgs else print('no renders found')

## GLB index
Every exported mesh. Open a file in any glTF viewer (for example by dragging it into https://gltf-viewer.donmccurdy.com, or into the viewer under `ui/`).

In [ ]:
glb = an.glb_index_table(OUT_A, OUT_B)
print(len(glb), 'GLB files')
show(glb[:200])